# Stage 3 · Finalize the dataset

Join accepted Stage 2 transcripts with their Stage 1 records, then export BTC scenarios, dataset splits, and audit reports.

In [ ]:
# Configure
from pathlib import Path
from runpy import run_path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import sys

BATCH_SIZE = 20
KAGGLE_INPUT = Path('/kaggle/input')

if KAGGLE_INPUT.is_dir():
    stages = [helper.parents[1] for helper in KAGGLE_INPUT.rglob('stage-3/src/finalize.py')]
    if not stages:
        archives = list(KAGGLE_INPUT.rglob('stage-3-kaggle-input.zip'))
        if len(archives) != 1:
            raise ValueError('Attach one Stage 3 Kaggle input dataset.')
        extracted = TemporaryDirectory(prefix='stage-3-input-')
        with ZipFile(archives[0]) as archive:
            archive.extractall(extracted.name)
        stages = [Path(extracted.name) / 'stage-3']
    if len(stages) != 1:
        raise ValueError('Attach only one Stage 3 Kaggle input dataset.')
    STAGE = stages[0]
    OUTPUT = Path('/kaggle/working/output')
else:
    STAGE = Path.cwd().resolve()
    if STAGE.name != 'stage-3':
        STAGE = STAGE / 'stage-3'
    OUTPUT = STAGE / 'output'

ROOT = STAGE.parent
INPUT = STAGE / 'input'
sys.path.insert(0, str(STAGE / 'src'))
finalize = run_path(str(STAGE / 'src/finalize.py'))['finalize']

In [ ]:
# Finalize
stats = finalize(
    INPUT, OUTPUT, batch_size=BATCH_SIZE, btc_dir=ROOT / 'btc',
    venue_catalog=ROOT / 'stage-1/input/catalog/public-delivery-addresses.json',
)

In [ ]:
# Review
print(f"{stats['tier']}: {stats['total_journeys']} journeys, {stats['total_sessions']} sessions")
print(f"Memory seed: {stats['memory_seed_journeys']} · Golden: {stats['golden_test_scenarios']} custom + {stats['btc_public_scenarios']} BTC")
print(f"Duplicates removed: {stats['duplicates_removed']} · Privacy rejections: {stats['privacy_rejections']}")
print(f"Outputs: {OUTPUT}")